# MGMT298D: Science and Strategy of AI
## Week 1: Linear Regression & Regularization
### UCLA Anderson School of Management

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import (LinearRegression, Lasso, Ridge, ElasticNet,
                                  LassoCV, RidgeCV, ElasticNetCV)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

## Load Data

In [ ]:
url = "https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/HMData.csv"
df = pd.read_csv(url)
print(f"{len(df)} rows, {df.shape[1]} columns")

# Show all columns in the preview
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
df.head()

## Filter by Product Type

In [ ]:
PRODUCT_TYPE = 'T-shirt'  # Change to any product name (see df['name'].unique())

df = df[df['name'] == PRODUCT_TYPE].copy()
print(f"{PRODUCT_TYPE}: {len(df)} rows, {df['id'].nunique()} products")

## Train/Test Split by Product
We hold out 20% of product IDs as our test set. The model never sees these products during training.

In [ ]:
np.random.seed(42)
product_ids = df['id'].unique()
np.random.shuffle(product_ids)
split = int(0.8 * len(product_ids))
train_ids, test_ids = product_ids[:split], product_ids[split:]

print(f"Train: {len(train_ids)} products, Test: {len(test_ids)} products")

---
# Block 1: Basic Linear Regression
Start simple — use only **price** and **month** to predict sales.

In [ ]:
# Months are already one-hot encoded in the CSV
month_cols = ['January', 'February', 'March', 'April', 'May', 'June',
              'July', 'August', 'September', 'October', 'November', 'December']

basic_features = ['price'] + month_cols
print(f"Features ({len(basic_features)}): {basic_features}")

In [ ]:
def split_and_scale(data, features, train_ids, test_ids):
    """Split by product ID, extract features, and standardize."""
    tr = data[data['id'].isin(train_ids)]
    te = data[data['id'].isin(test_ids)]
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(tr[features])
    X_te = scaler.transform(te[features])
    return X_tr, X_te, tr['sales'].values, te['sales'].values, scaler

X_train, X_test, y_train, y_test, scaler1 = split_and_scale(df, basic_features, train_ids, test_ids)

# Baseline: plain OLS with just price + month dummies
ols_basic = LinearRegression().fit(X_train, y_train)
mae_basic = mean_absolute_error(y_test, ols_basic.predict(X_test))
print(f"OLS (price + month) — Test MAE: {mae_basic:.1f}")

---
# Block 2: Feature Engineering → Overfitting Risk
Add lag features, rolling statistics, polynomial and interaction terms. With many features and plain OLS, we risk **overfitting** — the model memorizes training noise instead of learning real patterns.

In [ ]:
df_eng = df.copy()

# Lag features: sales from 1, 2, 3 months ago
df_eng['lag_1'] = df_eng.groupby('id')['sales'].shift(1)
df_eng['lag_2'] = df_eng.groupby('id')['sales'].shift(2)
df_eng['lag_3'] = df_eng.groupby('id')['sales'].shift(3)

# Rolling statistics (3-month window, shifted to avoid leakage)
df_eng['ma_3']  = df_eng.groupby('id')['sales'].transform(lambda x: x.rolling(3).mean().shift(1))
df_eng['std_3'] = df_eng.groupby('id')['sales'].transform(lambda x: x.rolling(3).std().shift(1))

# Price features
df_eng['price_pct_change'] = df_eng.groupby('id')['price'].pct_change()
df_eng['price_sq'] = df_eng['price'] ** 2

# Interaction terms
df_eng['price_x_lag_1'] = df_eng['price'] * df_eng['lag_1']
df_eng['lag1_x_lag2']   = df_eng['lag_1'] * df_eng['lag_2']

df_eng.fillna(0, inplace=True)
df_eng.replace([np.inf, -np.inf], 0, inplace=True)

# Explicit feature list (matches Week 1 app)
all_features = ['price', 'price_sq', 'price_pct_change',
                'lag_1', 'lag_2', 'lag_3', 'ma_3', 'std_3',
                'price_x_lag_1', 'lag1_x_lag2'] + month_cols
print(f"{len(all_features)} features: {all_features}")

In [ ]:
X_train2, X_test2, y_train2, y_test2, scaler2 = split_and_scale(df_eng, all_features, train_ids, test_ids)

# OLS with all engineered features — likely overfits
ols_eng = LinearRegression().fit(X_train2, y_train2)
mae_eng_train = mean_absolute_error(y_train2, ols_eng.predict(X_train2))
mae_eng_test  = mean_absolute_error(y_test2, ols_eng.predict(X_test2))

print(f"OLS ({len(all_features)} features)")
print(f"  Train MAE: {mae_eng_train:.1f}")
print(f"  Test  MAE: {mae_eng_test:.1f}")
print(f"  Gap: {mae_eng_test - mae_eng_train:.1f}  (large = overfitting)")

---
# Block 3: Regularization to the Rescue
Use **Lasso** (L1), **Ridge** (L2), and **Elastic Net** (L1+L2) to tame the large feature set.

- **λ (lambda)** controls the penalty strength for Lasso and Ridge
- **α (alpha)** is the Elastic Net mixing parameter: α=1 is pure Lasso, α=0 is pure Ridge

### Lasso (L1): Shrinks + eliminates coefficients

In [ ]:
# Sweep λ values for Lasso — higher λ = more coefficients pushed to zero
lambdas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
lasso_results = []

for lam in lambdas:
    m = Lasso(alpha=lam, max_iter=10000).fit(X_train2, y_train2)
    lasso_results.append({
        'λ': lam,
        'Train MAE': mean_absolute_error(y_train2, m.predict(X_train2)),
        'Test MAE':  mean_absolute_error(y_test2, m.predict(X_test2)),
        'nonzero':   int(np.sum(m.coef_ != 0))
    })

lasso_df = pd.DataFrame(lasso_results)
print(lasso_df.to_string(index=False))

### Ridge (L2): Shrinks coefficients but keeps them all

In [ ]:
# Sweep λ values for Ridge — shrinks coefficients but never zeros them out
ridge_results = []

for lam in lambdas:
    m = Ridge(alpha=lam).fit(X_train2, y_train2)
    ridge_results.append({
        'λ': lam,
        'Train MAE': mean_absolute_error(y_train2, m.predict(X_train2)),
        'Test MAE':  mean_absolute_error(y_test2, m.predict(X_test2)),
    })

ridge_df = pd.DataFrame(ridge_results)
print(ridge_df.to_string(index=False))

### Summary: Cross-Validated Model Comparison

In [ ]:
# Use cross-validation to select best λ for Lasso and Ridge, best λ + α for Elastic Net
lasso_cv  = LassoCV(cv=5, max_iter=10000, random_state=42).fit(X_train2, y_train2)
ridge_cv  = RidgeCV(cv=5, alphas=np.logspace(-3, 3, 50), scoring='neg_mean_absolute_error').fit(X_train2, y_train2)
enet_cv   = ElasticNetCV(cv=5, l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99],
                         max_iter=10000, random_state=42).fit(X_train2, y_train2)

print(f"LassoCV  best λ = {lasso_cv.alpha_:.4f}")
print(f"RidgeCV  best λ = {ridge_cv.alpha_:.4f}")
print(f"ElasticNetCV  best λ = {enet_cv.alpha_:.4f}, best α = {enet_cv.l1_ratio_:.2f}")

# Compare all models side by side
results = pd.DataFrame([
    {'Model': 'OLS (price + month)',           'Test MAE': mae_basic},
    {'Model': f'OLS ({len(all_features)} ft)', 'Test MAE': mae_eng_test},
    {'Model': f'Lasso (λ={lasso_cv.alpha_:.3f})',
     'Test MAE': mean_absolute_error(y_test2, lasso_cv.predict(X_test2))},
    {'Model': f'Ridge (λ={ridge_cv.alpha_:.3f})',
     'Test MAE': mean_absolute_error(y_test2, ridge_cv.predict(X_test2))},
    {'Model': f'ElasticNet (λ={enet_cv.alpha_:.3f}, α={enet_cv.l1_ratio_:.2f})',
     'Test MAE': mean_absolute_error(y_test2, enet_cv.predict(X_test2))},
])
print(results.to_string(index=False))

In [ ]:
colors = ['#888', '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bars = plt.bar(results['Model'], results['Test MAE'], color=colors, edgecolor='black', alpha=0.85)
for bar, val in zip(bars, results['Test MAE']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.0f}',
             ha='center', va='bottom', fontsize=11)
plt.ylabel('Test MAE')
plt.title('Test MAE Comparison')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()